# Selección Automática de Features usando Lasso Regularizado

En este notebook se realiza la selección automática de variables (features) a partir del dataset transformado `train` y `test`.
Se utiliza Lasso (`L1` regularization) de forma adaptable:
- Si el target es continuo (regresión): `sklearn.linear_model.Lasso`
- Si el target es binario: `sklearn.linear_model.LogisticRegression(penalty='l1', solver='liblinear')`

La regularización (`alpha` para Lasso, `C` para LogisticRegression) se ajusta automáticamente según el número de features para lograr un equilibrio entre reducción y retención de información.

---

## 1. Cargar los datos

Cargamos los datos de output del pipeline anterior.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.feature_selection import SelectFromModel
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Para reproducibilidad
def seed_everything(seed=42):
    import random, os
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
seed_everything()

# Ruta del directorio del pipeline
PIPELINE_DIR = 'proyecto_pipeline_machine_learning_muestra'

# Cargar datos transformados
X_train = pd.read_csv(os.path.join(PIPELINE_DIR, 'X_train.csv'))
X_test = pd.read_csv(os.path.join(PIPELINE_DIR, 'X_test.csv'))
y_train = pd.read_csv(os.path.join(PIPELINE_DIR, 'y_train.csv')).squeeze()
y_test = pd.read_csv(os.path.join(PIPELINE_DIR, 'y_test.csv')).squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, tipo: {y_train.dtype}")

---

## 2. Preparación del Target

- Si es continuo y está sesgado, se aplica log1p
- Si es categórico/binario, se mantiene para clasificación

In [ ]:
# Determinar si es regresión o clasificación
is_regression = (y_train.nunique() > 2)
print(f"Tipo de problema: {'Regresión' if is_regression else 'Clasificación binaria'}")

# Skew y log-transform si es necesario (solo para regresión)
if is_regression:
    skew = y_train.skew()
    print(f"Skew original y_train: {skew:.2f}")
    if abs(skew) > 0.75:
        y_train_sel = np.log1p(y_train)
        y_test_sel = np.log1p(y_test)
        print('Aplicado log1p al target.')
    else:
        y_train_sel = y_train.copy()
        y_test_sel = y_test.copy()
else:
    y_train_sel = y_train.copy()
    y_test_sel = y_test.copy()

---

## 3. Selección y ajuste del modelo Lasso apropiado

El hiperparámetro `alpha` se ajusta en función del número de features (más features, mayor regularización)

In [ ]:
n_features = X_train.shape[1]

# Definir alpha dinámico en función del número de features:
if n_features > 100:
    alpha = 0.01
else:
    alpha = 0.001
# Para clasificación binaria (LogisticRegression l1):
# C = inverso de regularización, más pequeño = más fuerte
C = 0.5 if n_features <= 100 else 0.2

print(f"n_features={n_features}, alpha usado para Lasso: {alpha if is_regression else '---'}, C para LogisticRegression: {C if not is_regression else '---'}")

# Ajustar modelo
if is_regression:
    model = Lasso(alpha=alpha, random_state=0, max_iter=5000)
else:
    model = LogisticRegression(penalty='l1', solver='liblinear', C=C, random_state=0, max_iter=1000)

model.fit(X_train, y_train_sel)

---

## 4. Visualización de coeficientes del modelo

Mostramos un barplot horizontal de importancias:
- Features eliminadas (coef=0): rojo
- Features seleccionadas (coef≠0): verde

In [ ]:
# Extracción de coeficientes
def get_coef(model):
    # Lasso
    if hasattr(model, 'coef_'):
        coef = model.coef_.ravel()
        return coef
    # LogisticRegression multinomial
    elif hasattr(model, 'coef_'):
        return model.coef_[0]
    else:
        raise ValueError('No se encontraron coeficientes en el modelo.')

coefs = get_coef(model)
features = X_train.columns

coef_df = pd.DataFrame({'feature': features, 'coef': coefs})
coef_df['abs_coef'] = coef_df['coef'].abs()
coef_df['selected'] = coef_df['coef'] != 0
coef_df = coef_df.sort_values('abs_coef', ascending=True)

col_palette = coef_df['selected'].map({True: 'forestgreen', False: 'firebrick'})

plt.figure(figsize=(10, max(6, n_features//10)))
plt.barh(coef_df['feature'], coef_df['coef'], color=col_palette)
plt.xlabel('Coeficiente')
plt.title('Coeficientes de Features tras Lasso / L1')
plt.tight_layout()
plt.show()

---

## 5. Selección de features con SelectFromModel

Extraemos features seleccionadas (coef distinto de cero) usando `SelectFromModel`.

In [ ]:
sfm = SelectFromModel(model, prefit=True, threshold=1e-12)  # Umbral muy bajo -> solo coef≠0
mask = sfm.get_support()
selected_features = features[mask].tolist()
removed_features = features[~mask].tolist()

print(f"Nº de features originales: {n_features}")
print(f"Nº de features seleccionadas: {len(selected_features)}")
print(f"Nº de features eliminadas: {len(removed_features)}\n")
print('Features seleccionadas:')
print(selected_features)

---

## 6. Exportar lista de features seleccionadas

Se exporta la lista a `.csv` compatible con el siguiente paso del pipeline.

In [ ]:
sel_path = os.path.join(PIPELINE_DIR, 'selected_features.csv')
pd.Series(selected_features).to_csv(sel_path, index=False)
print(f'✔️ Lista exportada a {sel_path}')

---

## 7. Conclusiones y Resumen

En este paso se han seleccionado automáticamente las mejores features para el modelo.
- Se empleó Lasso/L1 para eliminar coeficientes irrelevantes.
- El `alpha`/`C` fue adaptado automáticamente para ajustar la cantidad de selección vs. eliminación.
- Las features seleccionadas se exportan de manera reproducible para las siguientes etapas.

**Fin del notebook de selección automática de features.**